# Analisis de ecobici fecha 2024-04

## Librerias

In [1]:
from mongoengine import connect
import plotly.graph_objects as go
import plotly.express as px
from models import *
from bson import ObjectId
import pandas as pd
from requests import get
from datetime import datetime
from time import sleep
from os import getenv
from dotenv import load_dotenv

## Conexion con mongo atlas

In [2]:
load_dotenv()

client = connect('ecobiciCDMX',
        host=f"mongodb+srv://{getenv('MONGO_USER')}:{getenv('MONGO_KEY')}@{getenv('MONGO_APP_NAME').lower()}.eelkqty.mongodb.net/?retryWrites=true&w=majority&appName={getenv('MONGO_APP_NAME')}")

## Bussiness Understanding

### Vistazo general de los campos

In [5]:
first_document = Historico.objects.first().to_mongo().to_dict()
first_document

{'_id': ObjectId('680d7c6b80b27e0fdeb7fc2b'),
 'usuario': {'genero': 'M', 'edad': 32},
 'viaje': {'estacion_retiro': '198',
  'fecha_retiro': datetime.datetime(2024, 11, 30, 23, 35, 2),
  'estacion_arribo': '181',
  'fecha_arribo': datetime.datetime(2024, 12, 1, 0, 0, 1)}}

In [6]:
first_document = StationInformation.objects.first().to_mongo().to_dict()
first_document

{'_id': ObjectId('680d728d23614450558a60a1'),
 'station_id': '1',
 'name': 'CE-710 Molino del Rey - Glorieta de la Lealtad',
 'coords': {'type': 'Point', 'coordinates': [-99.192508, 19.416795]},
 'capacity': 39}

### Analisis por datos de usuario

#### Distribucion por genero

In [8]:
pipeline = [
    {
        '$project': {
            '_id': 0,
            'usuario.genero': {
                '$switch': {
                    'branches': [
                        {'case': {'$eq': ['$usuario.genero', 'M']}, 'then': 'Masculino'},
                        {'case': {'$eq': ['$usuario.genero', 'F']}, 'then': 'Femenino'},
                    ],
                    'default': 'Sin definir'
                }
            }
        }
    },
    {
        '$group': {
            '_id': '$usuario.genero',
            'cantidad': {'$count': {}}
        }
    },
    {
        '$project': {
            '_id': 0,
            'genero': '$_id',
            'cantidad': 1
        }
    },
    {
        '$sort': {
            'cantidad': -1
        }
    }
]

genero = pd.DataFrame(list(Historico.objects.aggregate(pipeline)))

fig = px.bar(genero, x='genero', y='cantidad',
            hover_data={'genero': False},
            labels={'genero': 'Genero', 'cantidad': 'Cantidad'},
            title='Distribucion de usuarios por genero',
            color='genero', template="plotly_dark")

fig.update_layout(
    title_x=0.5,
    showlegend=False
)

fig.show()

#### Distribucion por rangos de edad

In [9]:
pipeline = [
    {
        '$bucket': {
        'groupBy': "$usuario.edad",
        'boundaries': [0, 18, 35, 50, 65, 100],
        'default': "Fuera de rango",
        'output': {
            'cantidad': { '$count': {} }
        }}
    },
    {
        '$project': {
            '_id': 0,
            'Rango de edad': {
                '$switch': {
                    'branches': [
                        { 'case': { '$eq': ["$_id", 0] }, 'then': "0-18" },
                        { 'case': { '$eq': ["$_id", 18] }, 'then': "19-35" },
                        { 'case': { '$eq': ["$_id", 35] }, 'then': "36-50" },
                        { 'case': { '$eq': ["$_id", 50] }, 'then': "51-65" },
                        { 'case': { '$eq': ["$_id", 65] }, 'then': "66-100" },
                        { 'case': { '$eq': ["$_id", 'Fuera de rango'] }, 'then': "100+" }
                    ],
                    'default': "Fuera de rango"
                }
            },
            'cantidad': 1
        }
    }
]

edad = pd.DataFrame(list(Historico.objects.aggregate(pipeline)))
fig = px.bar(edad, x='Rango de edad', y='cantidad',
            hover_data={'Rango de edad': False},
            labels={'cantidad': 'Cantidad'},
            title='Distribucion de usuarios por rango de edades',
            color='Rango de edad', template="plotly_dark")

fig.update_layout(
    title_x=0.5,
    showlegend=False
)

fig.show()

#### Distribucion por edad y genero

In [10]:
pipeline = [
    {
        '$project': {
            '_id': 0,
            'usuario.genero': {
                '$switch': {
                    'branches': [
                        {'case': {'$eq': ['$usuario.genero', 'M']}, 'then': 'Masculino'},
                        {'case': {'$eq': ['$usuario.genero', 'F']}, 'then': 'Femenino'},
                    ],
                    'default': 'Sin_definir'
                }
            },
            'usuario.edad': 1
        }
    },
    {
        '$group': {
            '_id': {'genero': '$usuario.genero', 'edad': '$usuario.edad'},
            'cantidad': {'$count': {}}
        }
    },
    {
        "$project": {
            "_id": 0,
            "Rango de edad": {
            "$switch": {
                "branches": [
                    {
                        "case": {
                            "$and": [
                                { "$lt": ["$_id.edad", 18] },
                                { "$eq": ["$_id.genero", "Femenino"] }
                            ]
                        },
                        "then": "0-18 Femenino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$lt": ["$_id.edad", 18] },
                                { "$eq": ["$_id.genero", "Masculino"] }
                            ]
                        },
                        "then": "0-18 Masculino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$lt": ["$_id.edad", 18] },
                                { "$eq": ["$_id.genero", "Sin_definir"] }
                            ]
                        },
                        "then": "0-18 Sin_definir"
                    },

                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 18] },
                                { "$lt": ["$_id.edad", 36] },
                                { "$eq": ["$_id.genero", "Femenino"] }
                            ]
                        },
                        "then": "19-35 Femenino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 18] },
                                { "$lt": ["$_id.edad", 36] },
                                { "$eq": ["$_id.genero", "Masculino"] }
                            ]
                        },
                        "then": "19-35 Masculino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 18] },
                                { "$lt": ["$_id.edad", 36] },
                                { "$eq": ["$_id.genero", "Sin_definir"] }
                            ]
                        },
                        "then": "19-35 Sin_definir"
                    },

                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 36] },
                                { "$lt": ["$_id.edad", 51] },
                                { "$eq": ["$_id.genero", "Femenino"] }
                            ]
                        },
                        "then": "36-50 Femenino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 36] },
                                { "$lt": ["$_id.edad", 51] },
                                { "$eq": ["$_id.genero", "Masculino"] }
                            ]
                        },
                        "then": "36-50 Masculino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 36] },
                                { "$lt": ["$_id.edad", 51] },
                                { "$eq": ["$_id.genero", "Sin_definir"] }
                            ]
                        },
                        "then": "36-50 Sin_definir"
                    },

                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 51] },
                                { "$lt": ["$_id.edad", 66] },
                                { "$eq": ["$_id.genero", "Femenino"] }
                            ]
                        },
                        "then": "51-65 Femenino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 51] },
                                { "$lt": ["$_id.edad", 66] },
                                { "$eq": ["$_id.genero", "Masculino"] }
                            ]
                        },
                        "then": "51-65 Masculino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 51] },
                                { "$lt": ["$_id.edad", 66] },
                                { "$eq": ["$_id.genero", "Sin_definir"] }
                            ]
                        },
                        "then": "51-65 Sin_definir"
                    },

                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 66] },
                                { "$lte": ["$_id.edad", 100] },
                                { "$eq": ["$_id.genero", "Femenino"] }
                            ]
                        },
                        "then": "66-100 Femenino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 66] },
                                { "$lte": ["$_id.edad", 100] },
                                { "$eq": ["$_id.genero", "Masculino"] }
                            ]
                        },
                        "then": "66-100 Masculino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$gte": ["$_id.edad", 66] },
                                { "$lte": ["$_id.edad", 100] },
                                { "$eq": ["$_id.genero", "Sin_definir"] }
                            ]
                        },
                        "then": "66-100 Sin_definir"
                    },

                    {
                        "case": {
                            "$and": [
                                { "$gt": ["$_id.edad", 100] },
                                { "$eq": ["$_id.genero", "Femenino"] }
                            ]
                        },
                        "then": "100+ Femenino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$gt": ["$_id.edad", 100] },
                                { "$eq": ["$_id.genero", "Masculino"] }
                            ]
                        },
                        "then": "100+ Masculino"
                    },
                    {
                        "case": {
                            "$and": [
                                { "$gt": ["$_id.edad", 100] },
                                { "$eq": ["$_id.genero", "Sin_definir"] }
                            ]
                        },
                        "then": "100+ Sin_definir"
                    }
                    ],
                    "default": "Fuera de rango"
                }
            },
            "cantidad": 1
        }
    },
    {
        '$group': {
            '_id': '$Rango de edad',
            'cantidad': {'$sum': '$cantidad'}
        }
    },
    {
        '$project': {
            '_id': 0,
            'Rango de edad': {'$split': ['$_id', ' ']},
            'cantidad': 1
        }
    }
]

edadygenero = pd.DataFrame(list(Historico.objects.aggregate(pipeline)))
edadygenero['genero'] = edadygenero['Rango de edad'].apply(lambda x: x[1])
edadygenero['Rango de edad'] = edadygenero['Rango de edad'].apply(lambda x: x[0])
edadygenero["Rango de edad"] = pd.Categorical(edadygenero["Rango de edad"], categories=["0-18", "19-35", "36-50", "51-65", "66-100", "100+"], ordered=True)

fig = px.bar(
    edadygenero.sort_values(by='Rango de edad'),
    x="Rango de edad",
    y="cantidad",
    color="genero",
    barmode="group",
    color_discrete_map={
        "Femenino": "#E91E63",     # rosa
        "Masculino": "#2196F3",    # azul
        "Sin_definir": "#9E9E9E"   # gris
    },
    title="Distribución por Rango de Edad y Género",
    hover_data={'Rango de edad': False, 'genero': False},
    labels={'cantidad': 'Cantidad', 'genero': 'Genero'},
    template="plotly_dark"
)

fig.update_layout(
    title_x=0.5
)

fig.show()

### Analisis de los viajes

#### Distribucion por viajes iniciados

In [13]:
pipeline = [
    {
        '$project': {
            '_id': 0,
            'estacion_retiro': '$viaje.estacion_retiro',
            'fecha_retiro': '$viaje.fecha_retiro'
        }
    },
    # {
    #     '$match': {
    #         '$expr': {
    #             '$not': { '$regexMatch': { 'input': "$estacion_retiro", 'regex': "-" } }
    #         }
    #     }
    # },
    {
        '$group': {
            '_id': ['$estacion_retiro', {'$hour': '$fecha_retiro'}],
            'conteo': {'$count': {}}
        }
    },
    {
        '$project': {
            '_id': 0,
            'estacion_retiro': {'$arrayElemAt': ['$_id', 0]},
            'hora_retiro': {'$arrayElemAt': ['$_id', 1]},
            'conteo': 1
        }
    },
    {
        '$lookup': {
            'from': 'stationInformation',
            'localField': 'estacion_retiro',
            'foreignField': 'station_id',
            'as': 'estacion'
        }
    },
    {
        '$unwind': "$estacion"
    },
    {
        '$project': {
            'estacion_name': '$estacion.name',
            'hora_retiro': 1,
            'lat': {'$arrayElemAt': ['$estacion.coords.coordinates', 1]},
            'lon': {'$arrayElemAt': ['$estacion.coords.coordinates', 0]},
            'conteo': 1
        }
    },
    {
        '$sort': {
            'conteo': -1
        }
    },
]

viajes_retiro_hora = pd.DataFrame(list(Historico.objects.aggregate(pipeline)))
viajes_retiro_hora['marker_size'] = (viajes_retiro_hora['conteo'] - viajes_retiro_hora['conteo'].min()) / \
                                    (viajes_retiro_hora['conteo'].max() - viajes_retiro_hora['conteo'].min()) * \
                                    (30 - 5) + 5

# Crear los frames para cada hora
frames = []
for hour in sorted(viajes_retiro_hora['hora_retiro'].unique()):
    filtered_df = viajes_retiro_hora[viajes_retiro_hora['hora_retiro'] == hour]
    frame = go.Frame(
        data=[go.Scattermap(
            lat=filtered_df['lat'],
            lon=filtered_df['lon'],
            mode='markers',
            marker=dict(
                size=filtered_df['marker_size'],
                color=filtered_df['conteo'],
                cmin=viajes_retiro_hora['conteo'].min(),
                cmax=viajes_retiro_hora['conteo'].max(),
                showscale=True
            ),
            text=filtered_df['estacion_name'],
            hovertemplate='<b>Estacion:</b> %{text}<br>' +
                    '<b>Viajes iniciados:</b> %{customdata}<br>' +
                    '<extra></extra>',
            customdata=filtered_df['conteo']
        )],
        name=f'{hour}:00'
    )
    frames.append(frame)

fig = go.Figure(
    frames=frames,
    data=frames[0].data
)

# Agregar un slider para controlar las horas
fig.update_layout(
        updatemenus=[
        {
            'buttons': [
                {
                    'args': [None, {'frame': {'duration': 500, 'redraw': True}, 'fromcurrent': True}],
                    'label': 'Play',
                    'method': 'animate'
                },
                {
                    'args': [[None], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate', 'transition': {'duration': 0}}],
                    'label': 'Pause',
                    'method': 'animate'
                }
            ],
            'direction': 'left',
            'pad': {'r': 10, 't': 87},  # Espaciado de los botones
            'showactive': False,
            'type': 'buttons',
            'x': 0.5,  # Ubicación horizontal del botón
            'xanchor': 'center',
            'y': -0.1,  # Ubicación vertical del botón (puedes ajustar esto)
            'yanchor': 'top',
        }
    ],
    sliders=[{
        'steps': [
            {
                'args': [
                    [f'{hour}:00'],
                    {'frame': {'duration': 500, 'redraw': True}, 'mode': 'immediate', 'transition': {'duration': 0}}
                ],
                'label': f'{hour}:00',
                'method': 'animate'
            }
            for hour in sorted(viajes_retiro_hora['hora_retiro'].unique())
        ]
    }]
)

fig.update_layout(
    title={
        'text': 'Viajes iniciados por estacion',
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(color='white')
    },
    template="plotly_dark",
    map_style="open-street-map",
    map_zoom = 10,
    map_center={"lat": 19.42847, "lon": -99.12766},
)

fig.show()

#### Distribucion por viajes terminados

In [14]:
pipeline = [
    {
        '$project': {
            '_id': 0,
            'estacion_arribo': '$viaje.estacion_arribo',
            'fecha_arribo': '$viaje.fecha_arribo'
        }
    },
    # {
    #     '$match': {
    #         '$expr': {
    #             '$not': { '$regexMatch': { 'input': "$estacion_arribo", 'regex': "-" } }
    #         }
    #     }
    # },
    {
        '$group': {
            '_id': ['$estacion_arribo', {'$hour': '$fecha_arribo'}],
            'conteo': {'$count': {}}
        }
    },
    {
        '$project': {
            '_id': 0,
            'estacion_arribo': {'$arrayElemAt': ['$_id', 0]},
            'hora_arribo': {'$arrayElemAt': ['$_id', 1]},
            'conteo': 1
        }
    },
    {
        '$lookup': {
            'from': 'stationInformation',
            'localField': 'estacion_arribo',
            'foreignField': 'station_id',
            'as': 'estacion'
        }
    },
    {
        '$unwind': "$estacion"
    },
    {
        '$project': {
            'estacion_name': '$estacion.name',
            'hora_arribo': 1,
            'lat': {'$arrayElemAt': ['$estacion.coords.coordinates', 1]},
            'lon': {'$arrayElemAt': ['$estacion.coords.coordinates', 0]},
            'conteo': 1
        }
    },
    {
        '$sort': {
            'conteo': -1
        }
    },
]

viajes_arribo_hora = pd.DataFrame(list(Historico.objects.aggregate(pipeline)))
viajes_arribo_hora['marker_size'] = (viajes_arribo_hora['conteo'] - viajes_arribo_hora['conteo'].min()) / \
                                    (viajes_arribo_hora['conteo'].max() - viajes_arribo_hora['conteo'].min()) * \
                                    (30 - 5) + 5

# Crear los frames para cada hora
frames = []
for hour in sorted(viajes_arribo_hora['hora_arribo'].unique()):
    filtered_df = viajes_arribo_hora[viajes_arribo_hora['hora_arribo'] == hour]
    frame = go.Frame(
        data=[go.Scattermap(
            lat=filtered_df['lat'],
            lon=filtered_df['lon'],
            mode='markers',
            marker=dict(
                size=filtered_df['marker_size'],
                color=filtered_df['conteo'],
                cmin=viajes_arribo_hora['conteo'].min(),
                cmax=viajes_arribo_hora['conteo'].max(),
                showscale=True
            ),
            text=filtered_df['estacion_name'],
            hovertemplate='<b>Estacion:</b> %{text}<br>' +
                    '<b>Viajes terminados:</b> %{customdata}<br>' +
                    '<extra></extra>',
            customdata=filtered_df['conteo']
        )],
        name=f'{hour}:00'
    )
    frames.append(frame)

fig = go.Figure(
    frames=frames,
    data=frames[0].data
)

# Agregar un slider para controlar las horas
fig.update_layout(
        updatemenus=[
        {
            'buttons': [
                {
                    'args': [None, {'frame': {'duration': 500, 'redraw': True}, 'fromcurrent': True}],
                    'label': 'Play',
                    'method': 'animate'
                },
                {
                    'args': [[None], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate', 'transition': {'duration': 0}}],
                    'label': 'Pause',
                    'method': 'animate'
                }
            ],
            'direction': 'left',
            'pad': {'r': 10, 't': 87},  # Espaciado de los botones
            'showactive': False,
            'type': 'buttons',
            'x': 0.5,  # Ubicación horizontal del botón
            'xanchor': 'center',
            'y': -0.1,  # Ubicación vertical del botón (puedes ajustar esto)
            'yanchor': 'top',
        }
    ],
    sliders=[{
        'steps': [
            {
                'args': [
                    [f'{hour}:00'],
                    {'frame': {'duration': 500, 'redraw': True}, 'mode': 'immediate', 'transition': {'duration': 0}}
                ],
                'label': f'{hour}:00',
                'method': 'animate'
            }
            for hour in sorted(viajes_arribo_hora['hora_arribo'].unique())
        ]
    }]
)

fig.update_layout(
    title={
        'text': 'Viajes terminados por estacion',
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(color='white')
    },
    template="plotly_dark",
    map_style="open-street-map",
    map_zoom = 10,
    map_center={"lat": 19.42847, "lon": -99.12766},
)

fig.show()

#### Distribucion top 10 posibles rutas mas recorridas

In [16]:
pipeline = [
    {
        '$project': {
            '_id': 0,
            'estacion_retiro': '$viaje.estacion_retiro',
            'estacion_arribo': '$viaje.estacion_arribo'
        }
    },
    {
        '$match': {
            '$expr': {
                '$ne': ["$estacion_retiro", "$estacion_arribo"]
            }
        }
    },
    {
        '$group': {
        '_id': { 'estacion_retiro': "$estacion_retiro", 'estacion_arribo': '$estacion_arribo'},
        'conteo': { '$count': {} }
        }
    },
    {
        '$sort': {
            'conteo': -1
        }
    },
    {
        '$limit': 10
    },
    {
        '$project': {
            '_id': 0,
            'start': '$_id.estacion_retiro',
            'stop': '$_id.estacion_arribo',
            'conteo': 1
        }
    },
    {
        '$lookup': {
            'from': 'stationInformation',
            'localField': 'start',
            'foreignField': 'station_id',
            'as': 'start'
        }
    },
    {
        '$unwind': "$start"
    },
    {
        '$lookup': {
            'from': 'stationInformation',
            'localField': 'stop',
            'foreignField': 'station_id',
            'as': 'stop'
        }
    },
    {
        '$unwind': "$stop"
    },
    {
        '$project': {
            'start_name': '$start.name',
            'stop_name': '$stop.name',
            'start_lat': {'$arrayElemAt': ['$start.coords.coordinates', 1]},
            'start_lon': {'$arrayElemAt': ['$start.coords.coordinates', 0]},
            'stop_lat': {'$arrayElemAt': ['$stop.coords.coordinates', 1]},
            'stop_lon': {'$arrayElemAt': ['$stop.coords.coordinates', 0]},
            'conteo': 1
        }
    }
]

top10viajes = pd.DataFrame(list(Historico.objects.aggregate(pipeline)))
get_rutas = lambda coord: get(f"https://router.project-osrm.org/route/v1/bike/{coord['start_lon']},{coord['start_lat']};{coord['stop_lon']},{coord['stop_lat']}?overview=full&geometries=geojson").json()['routes'][0]['geometry']['coordinates']

fig = go.Figure()

for index, row in top10viajes.iterrows():
    coords = pd.DataFrame(get_rutas(row), columns=['lon', 'lat'])
    sleep(1)
    fig.add_trace(go.Scattermap(
        mode="lines",
        lat=coords['lat'],
        lon=coords['lon'],
        name=f'{row['start_name'][:6]} - {row['stop_name'][:6]}',
        line=dict(width=4),
        hovertemplate=(
            f"<b>Inicio:</b> {row['start_name']}<br>" +
            f"<b>Fin:</b> {row['stop_name']}<br>" +
            f"<b>Número de viajes:</b> {row['conteo']}<br>" +
            "<extra></extra>"
        ),
        visible=True if index == 0 else 'legendonly'
    ))

fig.update_layout(map_style="open-street-map",
                map_zoom = 10,
                map_center={"lat": 19.42847, "lon": -99.12766},
                title={
                    'text':'Top 10 posibles rutas mas recorridas',
                    'x': 0.5,
                    'xanchor': 'center',
                    'yanchor': 'top',
                    'font': dict(color='white') 
                },
                template="plotly_dark",
                showlegend=True
)

fig.show()

#### Meshplot distancia, media de edad, genero y conteo por genero y ruta

In [17]:
pipeline = [
    {
        '$project': {
            '_id': 0,
            'estacion_retiro': '$viaje.estacion_retiro',
            'estacion_arribo': '$viaje.estacion_arribo',
            'genero': {
                '$switch': {
                    'branches': [
                        {'case': {'$eq': ['$usuario.genero', 'M']}, 'then': 'Masculino'},
                        {'case': {'$eq': ['$usuario.genero', 'F']}, 'then': 'Femenino'},
                    ],
                    'default': 'Sin_definir'
                }
            },
            'edad': '$usuario.edad'
        }
    },
    {
        '$group': {
            '_id': { 'estacion_retiro': "$estacion_retiro",
                    'estacion_arribo': '$estacion_arribo',
                    'genero': '$genero'},
            'media_edad': {'$avg': '$edad'},
            'conteo': {'$count': {}}
        }
    },
    {
        '$sort': {
            'conteo': -1
        }
    },
    {
        '$project': {
            '_id': 0,
            'start_name': '$_id.estacion_retiro',
            'stop_name': '$_id.estacion_arribo',
            'genero': '$_id.genero',
            'media_edad': {'$toInt': '$media_edad'},
            'conteo': 1
        }
    }
]

get_station_id = lambda col, i: StationInformation.objects(name=top10viajes.loc[i,col]).first().to_mongo().to_dict()['station_id']

get_distance_trip = lambda coord: get(f"https://router.project-osrm.org/route/v1/bike/{coord['start_lon']},{coord['start_lat']};{coord['stop_lon']},{coord['stop_lat']}?overview=full&geometries=geojson").json()['routes'][0]['distance']


top10viajes_generos = pd.DataFrame(columns=['conteo', 'ruta', 'genero', 'media_edad', 'distancia'])

for i, row in top10viajes.iterrows():
    distancia = [get_distance_trip(row)]
    sleep(1)
    for doc in list(
            Historico.objects(
                viaje__estacion_retiro=get_station_id('start_name', i),
                viaje__estacion_arribo=get_station_id('stop_name', i)
            ).aggregate(pipeline)
        ):
        top10viajes_generos.loc[
            len(top10viajes_generos), :] = [doc['conteo']]+[row['start_name']+' -> '+row['stop_name']]\
                +list(doc.values())[3:]+distancia

top10viajes_generos['origen'] = top10viajes_generos['ruta'].str.split(' -> ').str[0]
top10viajes_generos['destino'] = top10viajes_generos['ruta'].str.split(' -> ').str[1]
top10viajes_generos['conteo'] = pd.to_numeric(top10viajes_generos['conteo'], errors='coerce')

fig = px.scatter_3d(
    top10viajes_generos,
    x='genero',
    y='media_edad',
    z='distancia',
    color='ruta',
    size='conteo',
    custom_data=['origen', 'destino', 'conteo']
)

fig.update_traces(
    hovertemplate=
        '<b>Start</b>: %{customdata[0]}<br>' +
        '<b>Stop</b>: %{customdata[1]}<br>' +
        '<b>Edad</b>: %{y}<br>' +
        '<b>Distancia [m]</b>: %{z}<br>' +
        '<b>Conteo</b>: %{customdata[2]}<br>' +
        '<extra></extra>'
)

fig.update_layout(
    template="plotly_dark",
    scene=dict(
        xaxis_title='Genero',
        yaxis_title='Edad Media',
        zaxis_title='Distancia [m]'
    ),
    title={
        'text':'Mesh plot de distancia de viaje, media de edad y genero',
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(color='white') 
    },
    showlegend=False
)

fig.show()

#### Histograma de viajes por hora del dia

In [18]:
pipeline = [
    {
        '$project': {
            'fechas': [
                {
                    'tipo': "retiro",
                    'hora': {'$hour': '$viaje.fecha_retiro'}
                },
                {
                    'tipo': "arribo",
                    'hora': {'$hour': "$viaje.fecha_arribo"}
                }
            ]
        }
    },
    { 
        '$unwind': "$fechas"
    },
    {
        '$group': {
            '_id': {
                'hora': "$fechas.hora",
                'tipo': "$fechas.tipo"
            },
            'conteo': { '$sum': 1 }
        }
    },
    {
        '$project': {
            '_id': 0,
            'hora': '$_id.hora',
            'tipo': '$_id.tipo',
            'conteo': 1
        }
    }
]

viajes_horas = pd.DataFrame(list(Historico.objects.aggregate(pipeline)))
fig = px.histogram(viajes_horas, x="hora", y='conteo',
                    color="tipo", nbins=24,
                    labels={
                        "hora": "Hora del Dia",
                        "conteo": "Viajes",
                        "tipo": "Tipo de Movimiento"
                    },
                    hover_data={'hora': True, 'conteo': True, 'tipo': True}
                )

fig.update_traces(opacity=0.6)

fig.update_layout(
    template="plotly_dark",
    yaxis_title="Numero de Viajes",
    title={
        'text':'Histograma de viajes por hora del dia',
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(color='white') 
    },
    showlegend=False,
    barmode='overlay'
)

fig.show()

#### Histograma de viajes por dia

In [19]:
pipeline = [
    {
        '$match': {
            '$expr': {
                '$gte': ["$viaje.fecha_retiro", datetime(2024, 12, 1) ]
            }
        }
    },
    {
        '$project': {
            'fechas': [
                {
                    'tipo': "retiro",
                    'fecha': {
                        '$dateTrunc': {
                            'date': '$viaje.fecha_retiro',
                            'unit': 'day'
                        }
                    }
                },
                {
                    'tipo': "arribo",
                    'fecha': {
                        '$dateTrunc': {
                            'date': '$viaje.fecha_arribo',
                            'unit': 'day'
                        }
                    }
                }
            ]
        }
    },
    { 
        '$unwind': "$fechas"
    },
    {
        '$group': {
            '_id': {
                'fecha': "$fechas.fecha",
                'tipo': "$fechas.tipo"
            },
            'conteo': { '$sum': 1 }
        }
    },
    {
        '$project': {
            '_id': 0,
            'fecha': '$_id.fecha',
            'tipo': '$_id.tipo',
            'conteo': 1
        }
    }
]

viajes_fecha = pd.DataFrame(list(Historico.objects.aggregate(pipeline)))
viajes_fecha['dia'] = viajes_fecha.fecha.apply(lambda x: x.day_name())

fig = px.histogram(viajes_fecha, x="fecha", y='conteo',
                    color="tipo", nbins=31,
                    labels={
                        "fecha": "Fecha",
                        "conteo": "Viajes",
                        "tipo": "Tipo de Movimiento",
                    },
                    hover_data={'fecha': True, 'conteo': True, 'tipo': True}
                )

fig.update_traces(opacity=0.6)

fig.update_layout(
    title={
        'text':'Histograma del numero de viajes cada dia',
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(color='white') 
    },
    template="plotly_dark",
    yaxis_title="Numero de Viajes",
    showlegend=False,
    barmode='overlay'
)

fig.show()